<a href="https://colab.research.google.com/github/AufanT/AufanT-BigData26_B_2411532011_AufanTaufiqurrahman/blob/main/Praktikum_2/BD_B_T02_2411532011_AufanTaufiqurrahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [3]:
from google.colab import drive
drive.mount("/content/drive")

df = pd.read_csv("/content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih.csv")
print("Dataset bersih SEED 42:", len(df), "baris")

Mounted at /content/drive
Dataset bersih SEED 42: 490 baris


In [4]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x.endswith(".0"):          # buang desimal ".0" SEBELUM titik ribuan dihapus
        x = x[:-2]
    x = x.replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

## Latihan 1 — Mengubah SEED menjadi 7
Pipeline K-2 s.d. K-6 dijalankan ulang dengan `SEED = 7`. Variabel dan file diberi akhiran `7` agar `transaksi_bersih.csv` milik SEED 42 tidak tertimpa.

In [5]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df7 = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df7.sample(frac=frac, random_state=SEED).index
    df7.loc[idx, col] = np.nan

dup_rows = df7.sample(n=15, random_state=SEED)
df7 = pd.concat([df7, dup_rows], ignore_index=True)
df7 = df7.sample(frac=1, random_state=SEED).reset_index(drop=True)

df7.to_csv("transaksi_mentah_seed7.csv", index=False)
print("Jumlah baris mentah (SEED 7):", len(df7))

Jumlah baris mentah (SEED 7): 515


In [6]:
print(df7.isnull().sum())
df7 = df7.dropna(subset=["customer_name", "payment_method"])
df7["shipping_city"] = df7["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah dropna():", len(df7))

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64
Jumlah baris setelah dropna(): 495


In [7]:
print("Baris duplicate (semua kolom sama):", df7.duplicated().sum())
df7 = df7.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df7))

Baris duplicate (semua kolom sama): 5
Jumlah baris setelah drop_duplicates(): 490


In [8]:
for col in ["category", "payment_method", "shipping_city"]:
    df7[col] = df7[col].astype("string").str.strip().str.title()
df7["payment_method"] = df7["payment_method"].replace({"Cod": "COD"})
df7["price"] = df7["price"].apply(bersihkan_harga)
df7["transaction_date"] = df7["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
df7["quantity"] = df7["quantity"].astype(int)
df7["price"] = df7["price"].astype(float)

In [9]:
df7.to_csv("transaksi_bersih_seed7.csv", index=False)
print("Dataset bersih tersimpan (SEED 7):", len(df7), "baris")

Dataset bersih tersimpan (SEED 7): 490 baris


In [10]:
print("\nPerbandingan:")
print("SEED 42 -> mentah: 515, bersih:", len(df))
print("SEED 7  -> mentah: 515, bersih:", len(df7))
print("\n5 baris pertama SEED 42:")
print(df.head())
print("\n5 baris pertama SEED 7:")
print(df7.head())


Perbandingan:
SEED 42 -> mentah: 515, bersih: 490
SEED 7  -> mentah: 515, bersih: 490

5 baris pertama SEED 42:
  transaction_id           customer_name product_name      category     price  \
0       TRX00305         Ophelia Hartati    Quo Basic          Buku   50000.0   
1       TRX00500      Cut Maya Wijayanti     Delectus    Elektronik  500000.0   
2       TRX00442  Dr. Ridwan Utama, M.Pd    Animi Max    Elektronik   25000.0   
3       TRX00154     Viman Suwarno, S.H.  Consectetur       Fashion  250000.0   
4       TRX00132      Ir. Ilyas Setiawan  Ducimus Pro  Rumah Tangga  250000.0   

   quantity payment_method transaction_date shipping_city  rating  
0         4   Kartu Kredit       2026-07-15        Blitar     1.0  
1         1       E-Wallet       2026-07-11  Lubuklinggau     NaN  
2         1            COD       2026-08-27   Probolinggo     2.0  
3         1            COD       2026-07-29          Tual     4.0  
4         4            COD       2026-09-17  Subulussalam   

**Analisis Latihan 1:**
Jumlah baris **sama** untuk kedua seed (mentah 515, bersih 490), tetapi **isi datanya berbeda** (nama pelanggan, produk, harga, dll.).

Alasannya:
1. Jumlah baris ditentukan oleh parameter yang tetap, bukan oleh seed: `N = 500`, 15 baris duplikat, dan fraksi missing value (0,02 / 0,03 / 0,015).
2. Karena ketiga `sample()` memakai `random_state` yang sama, baris yang diberi missing value pada kolom wajib selalu merupakan bagian dari baris yang diduplikasi, berapa pun seed-nya. Jadi pola pembuangannya selalu sama: 20 baris karena missing value dan 5 karena duplicate.
3. Seed hanya menentukan **baris mana** yang terpilih dan **nilai** datanya, bukan jumlahnya.

## Latihan 2 — Validasi Harga (`is_valid_price`)
Kolom `is_valid_price` bernilai `True` jika `price > 0`. Harga yang gagal diparsing (NaN) otomatis bernilai `False`.

In [11]:
df["is_valid_price"] = df["price"] > 0

print(df[["transaction_id", "price", "is_valid_price"]].head(10))
print("\nJumlah harga valid (True) dan tidak valid (False):")
print(df["is_valid_price"].value_counts())

  transaction_id      price  is_valid_price
0       TRX00305    50000.0            True
1       TRX00500   500000.0            True
2       TRX00442    25000.0            True
3       TRX00154   250000.0            True
4       TRX00132   250000.0            True
5       TRX00205   120000.0            True
6       TRX00010  1200000.0            True
7       TRX00326   150000.0            True
8       TRX00248  1200000.0            True
9       TRX00466    50000.0            True

Jumlah harga valid (True) dan tidak valid (False):
is_valid_price
True    490
Name: count, dtype: int64


**Hasil:** seluruh 490 transaksi bernilai `True`, artinya tidak ada harga nol, negatif, atau gagal diparsing.

**Keterbatasan:** pemeriksaan `price > 0` hanya mendeteksi harga yang tidak masuk akal secara tanda, bukan harga yang salah nilainya. Dengan fungsi `bersihkan_harga()` bawaan modul, format `"15000.0"` terbaca 150000 (10 kali lipat), dan harga yang salah ini tetap lolos pemeriksaan `price > 0`.

## Latihan 3 — Jumlah Transaksi per Kategori

In [12]:
print(df["category"].value_counts())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: int64


**Hasil:** terdapat **6 kategori unik** dengan total 490 transaksi. Kategori terbanyak adalah **Olahraga (97 transaksi)** dan tersedikit **Rumah Tangga (65 transaksi)**. Jumlah 6 kategori ini membuktikan standardisasi teks di K-5a berhasil. Tanpa standardisasi, nilai seperti "ELEKTRONIK  " akan terhitung sebagai kategori terpisah dari "Elektronik".